In [3]:
import polars as pl
import glob

In [2]:
# Configuration
INPUT_PATTERN = "/home/dnanexus/data_dir/bcf2parquet/*.parquet"  # Adjust path/pattern
OUTPUT_FILE = "/home/dnanexus/data_dir/bcf2parquet_consolidated_data.parquet"

# 1. Get list of files manually
files = glob.glob(INPUT_PATTERN)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
#    This builds a plan to read all files, but doesn't load them yet.
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
#    how="vertical_relaxed" allows slight type mismatches (like int32 vs int64)
#    how="diagonal" allows missing columns (fills with nulls)
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(OUTPUT_FILE, engine='streaming')
print("Done.")

Found 19396 files.
Streaming to disk...
Done.


## Check outfile

In [5]:
of = pl.scan_parquet(OUTPUT_FILE)
of.head().collect()

CHROM,POS,REF,ALT,SAMPLE,GT
str,str,str,str,str,i8
"""chr10""","""20020006""","""C""","""T""","""5317620""",1
"""chr10""","""20020007""","""TTTTCTTGC""","""T""","""5546988""",1
"""chr10""","""20020010""","""T""","""C""","""2793793""",1
"""chr10""","""20020013""","""T""","""G""","""1314475""",1
"""chr10""","""20020014""","""G""","""A""","""1722739""",1


In [6]:
(
    of
    .with_columns(
        POS = pl.col('POS').cast(pl.Int64)
    )
    .sink_parquet("/home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet", engine='streaming')
)

In [7]:
!dx upload /home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/

[===========================================================>] Uploaded 37,819,413,378 of 37,819,413,378 bytes (100%) /home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet=================================================>          ] Uploaded 31,406,948,352 of 37,819,413,378 bytes (83%) /home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet                                                     ] Uploaded 704,643,072 of 37,819,413,378 bytes (2%) /home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet[=====>                                                      ] Uploaded 3,623,878,656 of 37,819,413,378 bytes (10%) /home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet[========>                                                   ] Uploaded 5,838,471,168 of 37,819,413,378 bytes (15%) /home/dnanexus/data_dir/qced_maf1e-3_loftee_olink_genes.parquet[========>                                                   ] Uploaded 5,939,134,464 of 37,819,413,378 bytes (16%